## How to get a subset of Argo data for a heatwave

first load in our HW table from the ```MHW_list.csv``` file

In [ ]:
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np

# Path to Crocolake
parquet_dir = '/home/jovyan/shared/go-bgc-2026/data/CrocoLake/BGC_CROCOLAKE/'

# path to heatwave csv
csv_path = '../../MHW_list.csv'

dfhw = pd.read_csv(csv_path)
dfhw.set_index('name',inplace=True)
date_cols = ['date_start','date_end']
dfhw[date_cols] = dfhw[date_cols].apply(pd.to_datetime)

# Choose a HW
hw = "NA 2023"

# Boundaries
lat0 = dfhw.loc[hw,'lat0'] 
lat1 = dfhw.loc[hw,'lat1'] 
lon0 = dfhw.loc[hw,'lon0'] 
lon1 = dfhw.loc[hw,'lon1'] 

date0 = dfhw.loc[hw,'date_start']
date1 = dfhw.loc[hw,'date_end']


## CrocoLake Access

## Warning - subset doesn't work yet if crossing 180deg line 

In [ ]:
%%time

# Parameters
columns = ('PRES','TEMP','PSAL','DOXY','CHLA','BBP700','LATITUDE','LONGITUDE','JULD','CYCLE_NUMBER','DB_NAME','PLATFORM_NUMBER')


# without DB_NAME you also would get any Spray glider or shipboard GLODAP data too
filters = [
    ("LATITUDE",">",lat0), ("LATITUDE","<",lat1),
    ("LONGITUDE",">",lon0), ("LONGITUDE","<",lon1),
    ("JULD",">",date0), ("JULD","<",date1),
    ("DB_NAME","=","ARGO") 
]

df = pd.read_parquet(parquet_dir,columns=columns,filters=filters)
df = df.dropna(subset=['DOXY','CHLA','BBP700','PSAL','TEMP'])
df

In [ ]:
for wmo, data in df.groupby("PLATFORM_NUMBER"):
    # fig = plt.figure(figsize=(5, 5), constrained_layout=True)  # Increase height for extra row
    # ax = plt.subplot(1, 1, 1, projection=ccrs.Orthographic(
    # central_longitude=np.nanmean(data.LONGITUDE),
    # central_latitude=np.nanmean(data.LATITUDE)))
    # ax.set_global()
    # ax.coastlines()
    # ax.gridlines(draw_labels=True)

    # ax.scatter(data.LONGITUDE, data.LATITUDE, color='red', label='Float', s=10, marker='o', transform=ccrs.PlateCarree(), edgecolor='none')
    # fig.suptitle(f"Map of Argo data for {hw}")
    # plt.show()
    
    # # ax2 = plt.subplot(2, 3, 3)
    # # ax2.scatter(data.JULD, data.PRES, c=data.TEMP, label='Float')

    # # # plt.tight_layout()
    # # plt.show()
    fig = plt.figure(figsize=(10, 12), constrained_layout=True)
    ax1 = plt.subplot(3, 2, 1)
    ax1.scatter(data.TEMP, data.PRES, s=1, c = data.CYCLE_NUMBER)
    ax1.invert_yaxis()

    ax2 = plt.subplot(3, 2, 2)
    ax2.scatter(data.PSAL, data.PRES, s=1, c = data.CYCLE_NUMBER)
    ax2.invert_yaxis()
    
    ax3 = plt.subplot(3, 2, 3)
    ax3.scatter(data.DOXY, data.PRES, s=1, c = data.CYCLE_NUMBER)
    ax3.invert_yaxis()
    
    ax4 = plt.subplot(3, 2, 4)
    ax4.scatter(data.CHLA, data.PRES, s=1, c = data.CYCLE_NUMBER)
    ax4.invert_yaxis()

    ax5 = plt.subplot(3, 2, 5)
    ax5.scatter(data.BBP700, data.PRES, s=1, c = data.CYCLE_NUMBER)
    ax5.invert_yaxis()

    plt.show()
    break

## example for when lon bounds cross 180 deg line

In [ ]:
if lon0 > lon1:
    filters1 = [
    ("LATITUDE",">",lat0), ("LATITUDE","<",lat1),
    ("LONGITUDE",">",lon0), ("LONGITUDE","<=",180),
    ("JULD",">",date0), ("JULD","<",date1),
    ("DB_NAME","=","ARGO") 
    ]
    filters2 = [
    ("LATITUDE",">",lat0), ("LATITUDE","<",lat1),
    ("LONGITUDE",">",-180), ("LONGITUDE","<=",lon2),
    ("JULD",">",date0), ("JULD","<",date1),
    ("DB_NAME","=","ARGO") 
    ]
    df1 = pd.read_parquet(parquet_dir,columns=columns,filters=filters1)
    df2 = pd.read_parquet(parquet_dir,columns=columns,filters=filters2)
    df = pd.concat([df1, df2], ignore_index=True)
df = df.dropna(subset=['DOXY','CHLA','BBP700','PSAL','TEMP'])
df